# URL-only phishing warning model: Web Page only vs Web Page + PhiUSIIL

Same configuration as the revision experiments (MJCS-D-26-00034):
canonical URL (NFKC, scheme / leading `www.` / trailing `/` removed), registrable domain via tldextract,
5-fold CV grouped by registrable domain, seeds 42 and 44, the same six classifiers (LR + RF, SVM, DT, XGBoost,
LightGBM) and the same two inputs (34 lexical features; character 3-5-gram TF-IDF).

**Training setups**
* `WebPage`  : trained on Web Page URLs only.
* `Combined` : trained on Web Page + PhiUSIIL URLs. Each source gets the same total weight, so the 20x larger
  PhiUSIIL (legitimate = homepages only) cannot dominate.



In [ ]:
# ===================== 1. Setup =====================
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/input")
PLATFORM = "Colab" if IN_COLAB else ("Kaggle" if IN_KAGGLE else "local")
if IN_COLAB or IN_KAGGLE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tldextract"], check=False)
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
print("Platform:", PLATFORM)

import re, json, time, glob, pickle, platform, unicodedata, warnings, zipfile, shutil
from functools import lru_cache
from collections import OrderedDict
import numpy as np
import pandas as pd
import scipy
from scipy import stats
import matplotlib
if PLATFORM == "local":
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import sklearn, joblib
from importlib.metadata import version as _v
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, fbeta_score, roc_auc_score,
                             average_precision_score, matthews_corrcoef, confusion_matrix, roc_curve)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import tldextract
_TLD = tldextract.TLDExtract(suffix_list_urls=())   # bundled Public Suffix List snapshot -> offline, deterministic
warnings.filterwarnings("ignore")

In [ ]:
# ===================== 2. CONFIG (same files and settings as the revision notebook) =====================
DEFAULT_ROOT = {"Colab": "/content/drive/MyDrive", "Kaggle": "/kaggle/input", "local": "."}[PLATFORM]
DATA_DIR = os.environ.get("DATA_DIR", DEFAULT_ROOT)
FILES = {
    "PhiUSIIL": ["PhiUSIIL_Phishing_URL_Dataset.csv"],
    "WebPage":  ["Web_Page_Phishing_Detection.csv", "dataset_phishing.csv", "dataset_B_05_2020.csv"],
    "Ext1000":  ["Phishing_Detection_External_Evaluation_Dataset_1000_URLs.csv"],
}
FOLDER_HINTS = {
    "PhiUSIIL": ["phiusiil-phishing-url-dataset", "phiusiil"],
    "WebPage":  ["web-page-phishing-detection", "web-page-phishing"],
    "Ext1000":  ["phishing-detection-external-evaluation-dataset-100", "external-evaluation", "phishing-detection-external"],
}

def find_file(key, root):
    for n in FILES[key]:
        p = os.path.join(root, n)
        if os.path.exists(p):
            return p
        hits = sorted(glob.glob(os.path.join(root, "**", n), recursive=True))
        if hits:
            return hits[0]
    for hint in FOLDER_HINTS[key]:
        for folder in sorted(glob.glob(os.path.join(root, "**", "*" + hint + "*"), recursive=True)):
            if os.path.isdir(folder):
                csvs = sorted(glob.glob(os.path.join(folder, "**", "*.csv"), recursive=True))
                if csvs:
                    return csvs[0]
    raise FileNotFoundError(key + ": not found under " + root)

PATHS = {k: find_file(k, DATA_DIR) for k in FILES}
print(json.dumps(PATHS, indent=1))

OUT_DIR = os.environ.get("OUT_DIR", {"Colab": "/content/drive/MyDrive/url_guard_results",
                                     "Kaggle": "/kaggle/working/url_guard_results", "local": "url_guard_results"}[PLATFORM])
for sub in ["tables", "figures", "ckpt", "model", "repro"]:
    os.makedirs(os.path.join(OUT_DIR, sub), exist_ok=True)
TAB = lambda f: os.path.join(OUT_DIR, "tables", f)
FIG = lambda f: os.path.join(OUT_DIR, "figures", f)

SEEDS = [42, 44]
N_FOLDS = 5
SVM_MAX_TRAIN = 20000          # RBF-SVM: at most this many training rows per fold (as in the paper)
NGRAM_K = 2000                 # chi2-selected n-grams for RF, SVM, DT, XGBoost, LightGBM (as in the paper)
NGRAM_TREE_MAX_TRAIN = 60000   # RF and DT on n-grams: at most this many training rows (as in the paper)
N_DEEPLINK = 3000              # legitimate held-out URLs per fold used to build login / deep-link variants
N_LATENCY = 1000
MODELS = ["LR", "RF", "SVM", "DT", "XGBoost", "LightGBM"]
VIEWS = ["CanonURL", "CharNgram"]
SETUPS = ["WebPage", "Combined"]
QUICK = os.environ.get("QUICK", "0") == "1"   # local smoke test only
DEPLOY = "auto"                # or e.g. ("Combined", "CharNgram", "LR")
DEPLOY_CANDIDATES = ["LR", "RF", "XGBoost", "LightGBM"]   # give probabilities; SVM/DT are not used for the website
json.dump(dict(SEEDS=SEEDS, N_FOLDS=N_FOLDS, SVM_MAX_TRAIN=SVM_MAX_TRAIN, NGRAM_K=NGRAM_K,
               NGRAM_TREE_MAX_TRAIN=NGRAM_TREE_MAX_TRAIN, MODELS=MODELS, VIEWS=VIEWS, SETUPS=SETUPS,
               PATHS=PATHS, PLATFORM=PLATFORM), open(os.path.join(OUT_DIR, "repro", "config.json"), "w"), indent=2)

In [ ]:
# ===================== 3. URL utilities (identical to the revision notebook) =====================
UTIL_CODE = r'''
_SCHEME = re.compile(r"^[a-zA-Z][a-zA-Z0-9+.\-]*://")
_IP = re.compile(r"^\d{1,3}(\.\d{1,3}){3}$")
SENSITIVE = ("login", "signin", "sign-in", "logon", "verify", "verif", "secure", "account", "update", "confirm",
             "password", "passwd", "bank", "webscr", "auth", "wallet", "billing", "support", "unlock", "recover")
FILE_EXT = re.compile(r"\.(php|html?|aspx?|jsp|cgi|exe|zip|js)$")

def canonicalize(url):
    u = unicodedata.normalize("NFKC", str(url)).strip()
    u = re.sub(r"[\x00-\x20\x7f]", "", u)
    u = _SCHEME.sub("", u)
    m = re.match(r"([^/?#]*)(.*)", u)
    host, rest = m.group(1).lower().rstrip("."), m.group(2)
    if host.startswith("www."):
        host = host[4:]
    if rest == "/":
        rest = ""
    elif rest.endswith("/") and "?" not in rest and "#" not in rest:
        rest = rest.rstrip("/")
    return host + rest

def split_canon(c):
    m = re.match(r"([^/?#]*)(.*)", c)
    host, rest = m.group(1), m.group(2)
    userinfo = "@" in host
    hn = host.rsplit("@", 1)[-1]
    port = bool(re.search(r":\d+$", hn))
    hn = re.sub(r":\d+$", "", hn)
    return hn, rest, userinfo, port

@lru_cache(maxsize=200000)
def reg_parts(hn):
    if not hn or _IP.match(hn):
        return hn, ""
    e = _TLD(hn)
    rd = getattr(e, "top_domain_under_public_suffix", None) or getattr(e, "registered_domain", "")
    return (rd or hn), e.suffix

def _entropy(s):
    if not s:
        return 0.0
    _, c = np.unique(list(s), return_counts=True)
    p = c / c.sum()
    return float(-(p * np.log2(p)).sum())

def url_features(c):
    hn, rest, userinfo, port = split_canon(c)
    rd, suffix = reg_parts(hn)
    sub = hn[: -len(rd)].rstrip(".") if (rd and hn.endswith(rd) and hn != rd) else ""
    path, query = (rest.split("?", 1) + [""])[:2] if "?" in rest else (rest, "")
    path = path.split("#", 1)[0]
    low_host, low_rest = hn.lower(), rest.lower()
    digits = sum(ch.isdigit() for ch in c)
    special = sum((not ch.isalnum()) and ch not in "./" for ch in c)
    tokens = re.split(r"[./\-_?=&:]", c)
    return {
        "h_len": len(hn), "h_regdom_len": len(rd), "h_sub_len": len(sub),
        "h_sub_labels": 0 if not sub else sub.count(".") + 1, "h_dots": hn.count("."), "h_hyphens": hn.count("-"),
        "h_digits": sum(ch.isdigit() for ch in hn), "h_entropy": _entropy(hn), "h_is_ip": int(bool(_IP.match(hn))),
        "h_port": int(port), "h_userinfo": int(userinfo), "h_punycode": int("xn--" in hn), "h_tld_len": len(suffix),
        "h_sens_words": sum(w in low_host for w in SENSITIVE),
        "u_len": len(c), "u_digits": digits, "u_digit_ratio": digits / max(len(c), 1), "u_special": special,
        "u_hyphens": c.count("-"), "u_underscores": c.count("_"), "u_at": c.count("@"), "u_pct": c.count("%"),
        "u_tilde": c.count("~"), "u_longest_token": max((len(t) for t in tokens), default=0),
        "p_len": len(path), "p_depth": path.count("/"), "p_has_path": int(len(rest) > 0),
        "p_file_ext": int(bool(FILE_EXT.search(path.lower()))), "p_sens_words": sum(w in low_rest for w in SENSITIVE),
        "p_http_inside": int("http" in low_rest), "p_dslash": int("//" in rest),
        "q_len": len(query), "q_params": (query.count("&") + 1) if query else 0, "q_eq": query.count("="),
    }
'''
exec(UTIL_CODE, globals())

LEX_ALL = list(url_features("a.example.com/x?y=1").keys())

def lex_matrix(canon):
    return pd.DataFrame([url_features(c) for c in canon])[LEX_ALL].values.astype("float32")

In [ ]:
# ===================== 4. Data: PhiUSIIL + Web Page (training sources), D4V (external test only) ==========
def url_table(canon, y, src):
    t = pd.DataFrame({"canon": canon, "y": y})
    t = t[t["canon"].str.len() > 0]
    t = t[t.groupby("canon")["y"].transform("nunique") == 1].drop_duplicates("canon").reset_index(drop=True)
    t["dom"] = t["canon"].map(lambda c: reg_parts(split_canon(c)[0])[0])
    t["legit_path"] = (t["y"] == 0) & t["canon"].map(lambda c: len(split_canon(c)[1]) > 0)
    t["src"] = src
    return t

t0 = time.time()
_p = pd.read_csv(PATHS["PhiUSIIL"], encoding="utf-8-sig")
PHI = url_table(_p["URL"].map(canonicalize).values, (_p["label"] == 0).astype(int).values, "PhiUSIIL")   # 1 = phishing
_w = pd.read_csv(PATHS["WebPage"])
WEB = url_table(_w["url"].map(canonicalize).values, (_w["status"] == "phishing").astype(int).values, "WebPage")
_e = pd.read_csv(PATHS["Ext1000"])
D4V = url_table(_e["URL"].astype(str).map(canonicalize).values, (_e["Results"] == 1).astype(int).values, "D4V")
del _p, _w, _e
if QUICK:
    PHI, WEB = PHI.sample(4000, random_state=42).reset_index(drop=True), WEB.sample(3000, random_state=42).reset_index(drop=True)

# combined table: a canonical URL that occurs in both sources is kept once; conflicting labels are removed
ALL = pd.concat([WEB, PHI], ignore_index=True)
ALL = ALL[ALL.groupby("canon")["y"].transform("nunique") == 1].drop_duplicates("canon", keep="first").reset_index(drop=True)
D4V_TEST = D4V[~D4V["dom"].isin(set(ALL["dom"]))].reset_index(drop=True)   # same D4V subset for both setups
for name, t in [("Web Page", WEB), ("PhiUSIIL", PHI), ("Combined", ALL), ("D4V (all)", D4V), ("D4V (test)", D4V_TEST)]:
    print("%-11s %7d URLs | phishing %6d | domains %6d | legitimate deep links %5d"
          % (name, len(t), int(t.y.sum()), t.dom.nunique(), int(t.legit_path.sum())))
print("loaded in %.0fs" % (time.time() - t0))
pd.DataFrame({"canon": ALL["canon"], "src": ALL["src"], "y": ALL["y"]}).to_csv(
    os.path.join(OUT_DIR, "repro", "combined_urls.csv.gz"), index=False)

In [ ]:
# ===================== 5. Models, inputs, metrics, folds (same settings as the paper) =====================
def make_model(name, view, seed):
    if name == "LR":
        return (LogisticRegression(C=1.0, max_iter=2000, solver="liblinear") if view == "CharNgram"
                else make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=2000)))
    if name == "RF":
        return RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=seed)
    if name == "SVM":
        return (SVC(kernel="rbf", C=1.0, gamma="scale", random_state=seed) if view == "CharNgram"
                else make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0, gamma="scale", random_state=seed)))
    if name == "DT":
        return DecisionTreeClassifier(random_state=seed)
    if name == "XGBoost":
        return XGBClassifier(n_estimators=300, eval_metric="logloss", n_jobs=-1, random_state=seed)
    if name == "LightGBM":
        return LGBMClassifier(n_estimators=300, n_jobs=-1, random_state=seed, verbose=-1)
    raise ValueError(name)

def fit_weighted(model, X, y, w):
    if w is None:
        return model.fit(X, y)
    if isinstance(model, Pipeline):
        return model.fit(X, y, **{model.steps[-1][0] + "__sample_weight": w})
    return model.fit(X, y, sample_weight=w)

def scores(model, name, X):
    if name == "SVM":
        s = model.decision_function(X)
        return s, (s > 0).astype(int)
    s = model.predict_proba(X)[:, 1]
    return s, (s >= 0.5).astype(int)

def metrics(y, pred, score):
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    two = len(np.unique(y)) > 1
    fpr, fnr = fp / max(fp + tn, 1), fn / max(fn + tp, 1)
    return {"Acc": accuracy_score(y, pred), "Prec": precision_score(y, pred, zero_division=0),
            "Rec": recall_score(y, pred, zero_division=0), "F1": f1_score(y, pred, zero_division=0),
            "F2": fbeta_score(y, pred, beta=2, zero_division=0),
            "AUC": roc_auc_score(y, score) if two else np.nan,
            "PR_AUC": average_precision_score(y, score) if two else np.nan,
            "MCC": matthews_corrcoef(y, pred), "BAcc": 1 - (fpr + fnr) / 2, "FPR": fpr, "FNR": fnr,
            "n_test": int(len(y))}

class InputView:
    """CanonURL: 34 lexical features. CharNgram: char 3-5-gram TF-IDF fitted on the training rows only
    (LR uses all n-grams, the other models the chi2 top-NGRAM_K)."""
    def __init__(self, view):
        self.view = view
    def fit(self, canon, y):
        if self.view == "CharNgram":
            self.vec = TfidfVectorizer(analyzer="char", ngram_range=(3, 5), min_df=3, max_features=100000,
                                       sublinear_tf=True, dtype=np.float32).fit(canon)
            Xtr = self.vec.transform(canon)
            self.sel = SelectKBest(chi2, k=min(NGRAM_K, Xtr.shape[1])).fit(Xtr, y)
        return self
    def transform(self, canon, model_name):
        if self.view == "CanonURL":
            return lex_matrix(canon)
        X = self.vec.transform(list(canon))
        return X if model_name == "LR" else self.sel.transform(X).tocsr()

def source_weights(src):
    """Each training source gets the same total weight (mean weight = 1)."""
    src = np.asarray(src)
    n = len(src)
    counts = pd.Series(src).value_counts()
    return np.array([n / (len(counts) * counts[s]) for s in src], dtype=np.float64)

def balanced_cap(y, src, cap, seed):
    """Cap for SVM / RF / DT: equal rows from each source, stratified by class inside each source."""
    rs = np.random.RandomState(seed)
    srcs = np.unique(src)
    per = cap // len(srcs)
    pick = []
    for s in srcs:
        idx_s = np.where(src == s)[0]
        take = min(per, len(idx_s))
        for c in [0, 1]:
            idx = idx_s[y[idx_s] == c]
            k = int(round(take * len(idx) / len(idx_s)))
            pick.append(rs.choice(idx, min(k, len(idx)), replace=False))
    return np.sort(np.concatenate(pick))

def cv_splits_grouped(y, groups, seed):
    """Stratified K-fold over registrable domains (domain label = majority class) -> no domain in two folds."""
    codes = pd.Series(groups).astype("category").cat.codes.values
    glab = pd.Series(y).groupby(codes).mean().round().astype(int)
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    out = []
    for _, gte in skf.split(np.zeros(len(glab)), glab.values):
        te = np.isin(codes, glab.index.values[gte])
        out.append((np.where(~te)[0], np.where(te)[0]))
    return out

LOGIN_PATHS = ["/login", "/signin", "/account/login", "/user/login?next=%2Fdashboard", "/wp-login.php",
               "/auth/reset-password", "/my-account", "/customer/account/login", "/secure/checkout", "/verify-email"]
GENERIC_PATHS = ["/about", "/contact-us", "/en/products/item-1234", "/blog/2024/05/new-season", "/index.php?id=12",
                 "/help/faq", "/search?q=shoes&page=2", "/news/article.html", "/category/sale", "/terms"]
LOGIN_SUBS = ["login", "accounts", "secure", "my", "auth", "portal"]

def deeplink_variants(canon_legit, seed):
    rs = np.random.RandomState(seed)
    out = {"Login path": [], "Generic path": [], "Login subdomain": []}
    for c in canon_legit:
        hn = split_canon(c)[0]
        rd = reg_parts(hn)[0]
        out["Login path"].append(hn + rs.choice(LOGIN_PATHS))
        out["Generic path"].append(hn + rs.choice(GENERIC_PATHS))
        out["Login subdomain"].append(rs.choice(LOGIN_SUBS) + "." + rd)
    return out

In [ ]:
# ===================== 6. Paired CV: WebPage vs Combined (same folds, same test rows) =====================
def ckpt(*parts):
    return os.path.join(OUT_DIR, "ckpt", "__".join(str(p) for p in parts) + ".pkl")

y_all, src_all, canon_all = ALL["y"].values, ALL["src"].values, ALL["canon"].values
FOLDS = {seed: cv_splits_grouped(y_all, ALL["dom"].values, seed) for seed in SEEDS}
for seed in SEEDS:
    fid = np.full(len(ALL), -1)
    for k, (_, te) in enumerate(FOLDS[seed]):
        fid[te] = k
    pd.DataFrame({"canon": canon_all, "fold": fid}).to_csv(
        os.path.join(OUT_DIR, "repro", "folds_combined_seed%d.csv.gz" % seed), index=False)

t_start = time.time()
for view in VIEWS:
    for seed in SEEDS:
        for k, (tr_all, te) in enumerate(FOLDS[seed]):
            # identical test sets for both setups
            te_web = te[src_all[te] == "WebPage"]
            te_phi = te[src_all[te] == "PhiUSIIL"]
            rs = np.random.RandomState(seed * 100 + k)
            legit_phi = te_phi[y_all[te_phi] == 0]
            legit_phi = rs.choice(legit_phi, min(N_DEEPLINK, len(legit_phi)), replace=False)
            variants = deeplink_variants(canon_all[legit_phi], seed * 100 + k)
            tests = OrderedDict()
            tests["Web Page held-out"] = (canon_all[te_web], y_all[te_web], ALL["legit_path"].values[te_web])
            tests["PhiUSIIL held-out"] = (canon_all[te_phi], y_all[te_phi], ALL["legit_path"].values[te_phi])
            tests["D4V"] = (D4V_TEST["canon"].values, D4V_TEST["y"].values, D4V_TEST["legit_path"].values)
            for vname, urls in variants.items():
                tests["variant: " + vname] = (np.array(urls), np.zeros(len(urls), int), np.ones(len(urls), bool))
            for setup in SETUPS:
                cp = ckpt(setup, view, "s%d" % seed, "f%d" % k)
                if os.path.exists(cp):
                    continue
                tr = tr_all if setup == "Combined" else tr_all[src_all[tr_all] == "WebPage"]
                ytr, srctr = y_all[tr], src_all[tr]
                iv = InputView(view).fit(canon_all[tr], ytr)
                rows = []
                for m in MODELS:
                    cap = SVM_MAX_TRAIN if m == "SVM" else (NGRAM_TREE_MAX_TRAIN if (view == "CharNgram" and m in ("RF", "DT")) else None)
                    if cap and len(tr) > cap:
                        pick = balanced_cap(ytr, srctr, cap, seed)
                        idx, w = tr[pick], None                       # balanced sample -> no weights needed
                    else:
                        idx = tr
                        w = source_weights(srctr) if setup == "Combined" else None
                    Xtr = iv.transform(canon_all[idx], m)
                    mdl = make_model(m, view, seed)
                    t0 = time.perf_counter(); fit_weighted(mdl, Xtr, y_all[idx], w); t_fit = time.perf_counter() - t0
                    for tname, (c, yy, lp) in tests.items():
                        X = iv.transform(c, m)
                        t0 = time.perf_counter(); sc, pr = scores(mdl, m, X); t_inf = time.perf_counter() - t0
                        r = {"setup": setup, "view": view, "seed": seed, "fold": k, "model": m, "test": tname,
                             "n_train": len(idx), "n_train_web": int(np.sum(src_all[idx] == "WebPage")),
                             "n_train_phi": int(np.sum(src_all[idx] == "PhiUSIIL")), "train_s": t_fit,
                             "infer_ms_per_url": 1000 * t_inf / max(len(yy), 1)}
                        if tname.startswith("variant"):
                            r.update({"FPR": float(np.mean(pr == 1)), "n_test": len(yy)})
                        else:
                            r.update(metrics(yy, pr, sc))
                            r["FPR legit deep links"] = float(np.mean(pr[lp & (yy == 0)] == 1)) if (lp & (yy == 0)).any() else np.nan
                        rows.append(r)
                pickle.dump(rows, open(cp, "wb"))
                print("  %-8s %-9s seed=%d fold=%d done  (%.0f min)" % (setup, view, seed, k, (time.time() - t_start) / 60), flush=True)

R = pd.DataFrame([r for f in sorted(glob.glob(os.path.join(OUT_DIR, "ckpt", "*.pkl"))) for r in pickle.load(open(f, "rb"))])
R.to_csv(TAB("all_fold_results.csv"), index=False)
print(R.shape)

In [ ]:
# ===================== 7. Tables: mean +- SD and paired comparison =====================
def pm(s, scale=1.0, d=3):
    s = s.dropna() * scale
    return "%.*f $\\pm$ %.*f" % (d, s.mean(), d, s.std(ddof=1)) if len(s) else "--"

TESTS = ["Web Page held-out", "PhiUSIIL held-out", "D4V"]
summ = (R.groupby(["setup", "view", "model", "test"])[["Acc", "Prec", "Rec", "F1", "F2", "AUC", "PR_AUC", "MCC", "BAcc",
                                                       "FPR", "FNR", "FPR legit deep links", "train_s", "infer_ms_per_url"]]
          .agg(["mean", "std"]))
summ.to_csv(TAB("summary_mean_std.csv"))

# compact LaTeX table: ROC-AUC / MCC / FPR per test set + deep-link and variant FPR
lines = ["\\begin{tabular}{@{}lll" + "c" * 11 + "@{}}", "\\toprule",
         " & & & \\multicolumn{3}{c}{Web Page held-out} & \\multicolumn{3}{c}{PhiUSIIL held-out} & "
         "\\multicolumn{3}{c}{D4V} & \\multicolumn{2}{c}{Legit.\\ FPR (\\%)}\\\\",
         "Setup & Input & Model & ROC-AUC & MCC & FPR (\\%) & ROC-AUC & MCC & FPR (\\%) & ROC-AUC & MCC & FPR (\\%) & deep links & login subdom.\\\\",
         "\\midrule"]
for setup in SETUPS:
    for view in VIEWS:
        for m in MODELS:
            g = R[(R.setup == setup) & (R.view == view) & (R.model == m)]
            cells = []
            for t in TESTS:
                h = g[g.test == t]
                cells += [pm(h["AUC"]), pm(h["MCC"]), pm(h["FPR"], 100, 1)]
            dl = g[g.test == "Web Page held-out"]["FPR legit deep links"]
            ls = g[g.test == "variant: Login subdomain"]["FPR"]
            cells += [pm(dl, 100, 1), pm(ls, 100, 1)]
            lines.append(" & ".join([setup, view, m] + cells) + "\\\\")
        lines.append("\\midrule")
lines[-1] = "\\bottomrule"
lines.append("\\end{tabular}")
open(TAB("tab_combined.tex"), "w").write("\n".join(lines))

# paired comparison Combined - WebPage (Nadeau-Bengio corrected t-test over the 10 paired folds, Holm per test set)
def corrected_t(diff, n_tr, n_te):
    diff = np.asarray(diff, float); J = len(diff)
    if J < 2 or np.allclose(diff.std(ddof=1), 0):
        return np.nan, np.nan
    var = (1 / J + n_te / n_tr) * diff.var(ddof=1)
    t = diff.mean() / np.sqrt(var)
    return t, 2 * stats.t.sf(abs(t), J - 1)

def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); out = np.empty_like(p); run = 0
    for i, j in enumerate(o):
        run = max(run, min(1, (len(p) - i) * p[j])); out[j] = run
    return out

prow = []
for view in VIEWS:
    for t in TESTS + ["variant: Login path", "variant: Generic path", "variant: Login subdomain"]:
        metric = "FPR" if t.startswith("variant") else "AUC"
        for m in MODELS:
            a = R[(R.setup == "Combined") & (R.view == view) & (R.model == m) & (R.test == t)].sort_values(["seed", "fold"])
            b = R[(R.setup == "WebPage") & (R.view == view) & (R.model == m) & (R.test == t)].sort_values(["seed", "fold"])
            if len(a) != len(b) or len(a) == 0:
                continue
            d = a[metric].values - b[metric].values
            tt, p = corrected_t(d, a["n_train"].mean(), a["n_test"].mean())
            prow.append({"view": view, "test": t, "metric": metric, "model": m, "WebPage": b[metric].mean(),
                         "Combined": a[metric].mean(), "diff": d.mean(), "p": p})
P = pd.DataFrame(prow)
P["p_holm"] = np.nan
for (v, t), g in P.groupby(["view", "test"]):
    P.loc[g.index, "p_holm"] = holm(g["p"].fillna(1).values)
P.to_csv(TAB("paired_combined_vs_webpage.csv"), index=False)
pd.set_option("display.width", 250)
print(P.round(4).to_string(index=False))

In [ ]:
# ===================== 8. Figures =====================
COL = {"WebPage": "#1f77b4", "Combined": "#c0392b"}
fig, axes = plt.subplots(len(VIEWS), len(TESTS), figsize=(15, 7), sharey=True)
for i, view in enumerate(VIEWS):
    for j, t in enumerate(TESTS):
        ax = axes[i, j]
        x = np.arange(len(MODELS))
        for s_i, setup in enumerate(SETUPS):
            g = R[(R.setup == setup) & (R.view == view) & (R.test == t)].groupby("model")["AUC"]
            mu, sd = g.mean().reindex(MODELS), g.std().reindex(MODELS)
            ax.bar(x + (s_i - 0.5) * 0.38, mu, 0.38, yerr=sd, color=COL[setup], label=setup, capsize=2)
        ax.set_xticks(x); ax.set_xticklabels(MODELS, rotation=30, fontsize=8); ax.set_ylim(0.5, 1.0)
        ax.axhline(0.5, ls=":", c="grey")
        ax.set_title("%s | %s" % (view, t), fontsize=9)
        if j == 0:
            ax.set_ylabel("ROC-AUC")
axes[0, 0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIG("fig_combined_auc.pdf")); plt.savefig(FIG("fig_combined_auc.png"), dpi=150); plt.show()

VT = ["Web Page held-out", "variant: Login path", "variant: Generic path", "variant: Login subdomain"]
fig, axes = plt.subplots(1, len(VIEWS), figsize=(13, 3.8), sharey=True)
for i, view in enumerate(VIEWS):
    ax = axes[i]
    x = np.arange(len(VT))
    for s_i, setup in enumerate(SETUPS):
        vals, errs = [], []
        for t in VT:
            col = "FPR legit deep links" if t == "Web Page held-out" else "FPR"
            g = R[(R.setup == setup) & (R.view == view) & (R.test == t)].groupby("model")[col].mean() * 100
            vals.append(g.min()); errs.append(g.max())
        ax.bar(x + (s_i - 0.5) * 0.38, errs, 0.38, color=COL[setup], alpha=0.35)
        ax.bar(x + (s_i - 0.5) * 0.38, vals, 0.38, color=COL[setup], label=setup + " (best model / worst model)")
    ax.set_xticks(x); ax.set_xticklabels(["real deep links", "login path", "generic path", "login subdomain"], fontsize=8)
    ax.set_title(view, fontsize=9)
    if i == 0:
        ax.set_ylabel("legitimate URLs flagged (%)")
axes[0].legend(fontsize=7)
plt.tight_layout(); plt.savefig(FIG("fig_combined_deeplink_fpr.pdf")); plt.savefig(FIG("fig_combined_deeplink_fpr.png"), dpi=150); plt.show()

In [ ]:
# ===================== 9. Choose the website model and retrain it on all data =====================
cand = R[R.model.isin(DEPLOY_CANDIDATES) & R.test.isin(TESTS)].groupby(["setup", "view", "model", "test"])["AUC"].mean().unstack()
cand["mean_AUC"] = cand[TESTS].mean(axis=1)
dl = R[R.test == "Web Page held-out"].groupby(["setup", "view", "model"])["FPR legit deep links"].mean()
cand["deep_link_FPR"] = dl.reindex(cand.index)
cand = cand.sort_values("mean_AUC", ascending=False)
print(cand.round(4).to_string())
cand.to_csv(TAB("deploy_candidates.csv"))
if DEPLOY == "auto":
    DEPLOY = tuple(cand.index[0])
setup, view, mname = DEPLOY
print("Website model:", DEPLOY)

tr = np.arange(len(ALL)) if setup == "Combined" else np.where(src_all == "WebPage")[0]
iv = InputView(view).fit(canon_all[tr], y_all[tr])
w = source_weights(src_all[tr]) if setup == "Combined" else None
final = make_model(mname, view, SEEDS[0])
fit_weighted(final, iv.transform(canon_all[tr], mname), y_all[tr], w)

bundle = {"setup": setup, "view": view, "model_name": mname, "model": final,
          "vectorizer": getattr(iv, "vec", None), "selector": getattr(iv, "sel", None),
          "lex_features": LEX_ALL, "threshold": 0.5, "bands": {"low": 0.3, "high": 0.7},
          "trained_on": {"n": int(len(tr)), "WebPage": int(np.sum(src_all[tr] == "WebPage")),
                         "PhiUSIIL": int(np.sum(src_all[tr] == "PhiUSIIL"))},
          "cv": cand.loc[DEPLOY].to_dict(),
          "versions": {"python": platform.python_version(), "sklearn": sklearn.__version__, "numpy": np.__version__,
                       "pandas": pd.__version__, "xgboost": _v("xgboost"), "lightgbm": _v("lightgbm"), "tldextract": _v("tldextract")}}
MODEL_PATH = os.path.join(OUT_DIR, "model", "url_guard_model.joblib")
joblib.dump(bundle, MODEL_PATH, compress=3)
print("saved", MODEL_PATH, "%.1f MB" % (os.path.getsize(MODEL_PATH) / 1e6))
json.dump({k: v for k, v in bundle.items() if k in ("setup", "view", "model_name", "threshold", "bands", "trained_on", "cv", "versions")},
          open(os.path.join(OUT_DIR, "model", "model_card.json"), "w"), indent=2, default=float)

In [ ]:
# ===================== 11. Zip everything =====================
zpath = os.path.join(os.path.dirname(OUT_DIR), "url_guard_results.zip")
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT_DIR):
        for f in files:
            full = os.path.join(root, f)
            if "/ckpt/" in full.replace("\\", "/"):
                continue
            z.write(full, os.path.relpath(full, os.path.dirname(OUT_DIR)))
print("download:", zpath)